In [1]:
import pandas as pd
import numpy as np
import os

# Load your saved master table
data_folder = os.path.expanduser("~/Desktop/gcc-wef-nexus/data")
df = pd.read_csv(os.path.join(data_folder, "gcc_master_clean.csv"))

print("Data loaded successfully!")
print("Shape:", df.shape)
print("\nFirst 3 rows:")
df.head(3)

Data loaded successfully!
Shape: (144, 9)

First 3 rows:


,country,code,year,electricity_access,food_imports,freshwater_per_capita,renewable_electricity,undernourishment,water_withdrawal
0,Bahrain,BHR,2000,100.0,9.698643,6.273703,0.0,NaN,7218.413475
1,Bahrain,BHR,2001,100.0,11.741891,6.048546,0.0,NaN,6808.108975
2,Bahrain,BHR,2002,100.0,10.935628,5.629418,0.0,NaN,6397.804500


In [2]:
# Focus on most recent 5 years for the index
recent = df[df['year'] >= 2018].copy()

# Average each indicator per country over 2018-2023
country_avg = recent.groupby(['country', 'code']).agg({
    'electricity_access': 'mean',
    'food_imports': 'mean',
    'freshwater_per_capita': 'mean',
    'renewable_electricity': 'mean',
    'undernourishment': 'mean',
    'water_withdrawal': 'mean'
}).reset_index()

print("Country averages (2018-2023):")
print(country_avg.round(2).to_string())

Country averages (2018-2023):
                country code  electricity_access  food_imports  freshwater_per_capita  renewable_electricity  undernourishment  water_withdrawal
0               Bahrain  BHR              100.00         11.84                   2.67                   0.03               NaN           3877.50
1                Kuwait  KWT              100.00         17.17                   0.00                   0.13              2.50               NaN
2                  Oman  OMN              100.00         15.45                 305.21                   0.66              5.92            116.71
3                 Qatar  QAT              100.00         11.23                  21.28                   0.28               NaN            446.43
4          Saudi Arabia  SAU               99.98         14.93                  77.59                   0.06              2.50            974.17
5  United Arab Emirates  ARE              100.00          6.04                  15.69               

In [3]:
# We will score each indicator 0-10
# 10 = most vulnerable, 0 = least vulnerable

# Make a working copy
index_df = country_avg.copy()

# Fill NaN undernourishment with 2.5 
# (FAO standard: NaN means below 2.5% threshold)
index_df['undernourishment'] = index_df['undernourishment'].fillna(2.5)

# Fill NaN water_withdrawal with the group median
index_df['water_withdrawal'] = index_df['water_withdrawal'].fillna(
    index_df['water_withdrawal'].median()
)

print("NaN values remaining:")
print(index_df.isnull().sum())
print("\nData ready for normalisation:")
print(index_df.round(2).to_string())

NaN values remaining:
country                  0
code                     0
electricity_access       0
food_imports             0
freshwater_per_capita    0
renewable_electricity    0
undernourishment         0
water_withdrawal         0
dtype: int64

Data ready for normalisation:
                country code  electricity_access  food_imports  freshwater_per_capita  renewable_electricity  undernourishment  water_withdrawal
0               Bahrain  BHR              100.00         11.84                   2.67                   0.03              2.50           3877.50
1                Kuwait  KWT              100.00         17.17                   0.00                   0.13              2.50            974.17
2                  Oman  OMN              100.00         15.45                 305.21                   0.66              5.92            116.71
3                 Qatar  QAT              100.00         11.23                  21.28                   0.28              2.50            

In [4]:
def normalise(series, higher_is_worse=True):
    """
    Normalise a series to 0-10 scale.
    If higher_is_worse=True: high value = high vulnerability score
    If higher_is_worse=False: high value = low vulnerability score
    """
    min_val = series.min()
    max_val = series.max()
    
    if higher_is_worse:
        # Higher raw value = higher vulnerability
        normalised = (series - min_val) / (max_val - min_val) * 10
    else:
        # Lower raw value = higher vulnerability (reverse)
        normalised = (max_val - series) / (max_val - min_val) * 10
    
    return normalised

# Now score each indicator
# Higher water_withdrawal = MORE vulnerable
index_df['score_water_withdrawal'] = normalise(
    index_df['water_withdrawal'], higher_is_worse=True
)

# Lower freshwater_per_capita = MORE vulnerable
index_df['score_freshwater'] = normalise(
    index_df['freshwater_per_capita'], higher_is_worse=False
)

# Higher food_imports = MORE vulnerable
index_df['score_food_imports'] = normalise(
    index_df['food_imports'], higher_is_worse=True
)

# Higher undernourishment = MORE vulnerable
index_df['score_undernourishment'] = normalise(
    index_df['undernourishment'], higher_is_worse=True
)

# Lower renewable_electricity = MORE vulnerable
index_df['score_renewable'] = normalise(
    index_df['renewable_electricity'], higher_is_worse=False
)

# Lower electricity_access = MORE vulnerable
index_df['score_electricity_access'] = normalise(
    index_df['electricity_access'], higher_is_worse=False
)

print("Individual vulnerability scores (0-10, higher = more vulnerable):")
score_cols = ['country', 'score_water_withdrawal', 'score_freshwater', 
              'score_food_imports', 'score_undernourishment', 
              'score_renewable', 'score_electricity_access']

print(index_df[score_cols].round(2).to_string())

Individual vulnerability scores (0-10, higher = more vulnerable):
                country  score_water_withdrawal  score_freshwater  score_food_imports  score_undernourishment  score_renewable  score_electricity_access
0               Bahrain                   10.00              9.91                5.21                    0.00            10.00                       0.0
1                Kuwait                    2.28             10.00               10.00                    0.00             9.68                       0.0
2                  Oman                    0.00              0.00                8.46                   10.00             7.81                       0.0
3                 Qatar                    0.88              9.30                4.66                    0.00             9.14                       0.0
4          Saudi Arabia                    2.28              7.46                7.99                    0.00             9.89                      10.0
5  United Arab E

In [5]:
# Combine into three component scores
# WATER component = average of water indicators
index_df['water_score'] = index_df[[
    'score_water_withdrawal', 
    'score_freshwater'
]].mean(axis=1)

# ENERGY component = average of energy indicators  
index_df['energy_score'] = index_df[[
    'score_renewable', 
    'score_electricity_access'
]].mean(axis=1)

# FOOD component = average of food indicators
index_df['food_score'] = index_df[[
    'score_food_imports', 
    'score_undernourishment'
]].mean(axis=1)

# FINAL WEF INDEX = weighted average
# Water 35%, Energy 35%, Food 30%
index_df['wef_index'] = (
    index_df['water_score'] * 0.35 +
    index_df['energy_score'] * 0.35 +
    index_df['food_score'] * 0.30
)

# Rank countries (1 = most vulnerable)
index_df['rank'] = index_df['wef_index'].rank(ascending=False).astype(int)

# Show final results
final = index_df[['country', 'water_score', 'energy_score', 
                   'food_score', 'wef_index', 'rank']].sort_values('rank')

print("=" * 65)
print("   GCC WATER-ENERGY-FOOD NEXUS VULNERABILITY INDEX 2018-2023")
print("=" * 65)
print(final.round(2).to_string(index=False))
print("\n10 = most vulnerable | 0 = least vulnerable")

   GCC WATER-ENERGY-FOOD NEXUS VULNERABILITY INDEX 2018-2023
             country  water_score  energy_score  food_score  wef_index  rank
        Saudi Arabia         4.87          9.95        3.99       6.38     1
             Bahrain         9.96          5.00        2.60       6.02     2
              Kuwait         6.14          4.84        5.00       5.34     3
                Oman         0.00          3.90        9.23       4.13     4
               Qatar         5.09          4.57        2.33       4.08     5
United Arab Emirates         6.70          0.00        0.17       2.39     6

10 = most vulnerable | 0 = least vulnerable


In [6]:
# Save the full index results
index_df.to_csv(
    os.path.join(data_folder, "gcc_wef_index_results.csv"),
    index=False
)

# Save the clean final table
final.to_csv(
    os.path.join(data_folder, "gcc_wef_final_rankings.csv"),
    index=False
)

print("✓ Full index results saved: gcc_wef_index_results.csv")
print("✓ Final rankings saved: gcc_wef_final_rankings.csv")
print("\nPhase 3 complete. Ready for visualisation.")

✓ Full index results saved: gcc_wef_index_results.csv
✓ Final rankings saved: gcc_wef_final_rankings.csv

Phase 3 complete. Ready for visualisation.
